# Chapter 13: Continuous Deployment: Environments and OIDC (notebook edition)

## Learning Objectives

- Read deployment histories
- Take a real OIDC token apart
- Evaluate trust policies against real claims

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Deployment status history

The approved production deploy went waiting, queued, in_progress, success; the deploy from a feature branch went waiting, then failure.

In [ ]:
import json
s = json.loads((ROOT/"fixtures"/"deploy_ch13_summary.json").read_text())
dep, oidc = s["deployments"], s["oidc"]
print(dep["approved"]["statuses"]); print(dep["branch_restriction"]["statuses"]); print(dep["branch_restriction"]["annotations"][0])
assert dep["approved"]["statuses"][-1] == "success" and dep["branch_restriction"]["statuses"][-1] == "failure"

## 2. The real token

Signature valid, 300-second lifetime, wrong audience and a tampered signature both rejected.

In [ ]:
print(oidc["alg"], oidc["lifetime_seconds"], oidc["wrong_audience"], oidc["tampered_signature"], len(oidc["claim_names"]), "claims")
assert oidc["lifetime_seconds"] == 300 and len(oidc["claim_names"]) == 33
iat, life = 1000, 300
print("valid at 1250:", 1250 < iat + life, "| valid at 1400:", 1400 < iat + life)

## 3. The `sub` claim four ways

With an environment the sub ignores the branch; without one it names the ref.

In [ ]:
c = oidc["claims"]
for k, v in c.items():
    print(f"{k:28} {v['sub'].split(':', 2)[2]:36} ref={v['ref']}")
assert c["with_environment_main"]["sub"] == c["with_environment_branch"]["sub"]
assert c["without_environment_main"]["sub"] != c["without_environment_branch"]["sub"]

## 4. Evaluate trust policies

Our model checks `iss`, `aud`, then `sub` with `*` wildcards.

In [ ]:
from intro_gha.oidc import immutable_sub, legacy_sub, trust_allows
iss, aud = oidc["issuer"], "sts.amazonaws.com"
tok = lambda k: {"iss": iss, "aud": aud, "sub": c[k]["sub"]}
R = ("Friend09", 7501015, "practice_git_intro_to_github_actions", 1405684500)
cases = {
 "env=production vs staging token": ({"iss": iss, "aud": aud, "sub": immutable_sub(*R, "environment:production")}, "with_environment_main"),
 "legacy-format policy": ({"iss": iss, "aud": aud, "sub": legacy_sub("Friend09", R[2], "environment:staging")}, "with_environment_main"),
 "ref=main on main": ({"iss": iss, "aud": aud, "sub": immutable_sub(*R, "ref:refs/heads/main")}, "without_environment_main"),
 "ref=main on feature branch": ({"iss": iss, "aud": aud, "sub": immutable_sub(*R, "ref:refs/heads/main")}, "without_environment_branch"),
 "wildcard on feature branch": ({"iss": iss, "aud": aud, "sub": immutable_sub(*R, "*")}, "with_environment_branch"),
}
for name, (policy, key) in cases.items():
    print(f"{name:34}", trust_allows(tok(key), policy))
assert not trust_allows(tok("with_environment_main"), cases["legacy-format policy"][0])[0]
assert trust_allows(tok("with_environment_branch"), cases["wildcard on feature branch"][0])[0]

## Takeaways & Next Steps

- Put branch rules on the environment.
- Grant `id-token: write` to one job.
- Check `aud` and `sub`; mind the new ID-based `sub`.
- Next: Chapter 14, release automation.

## Chapter Link

Read: `learning_modules/chapter_13_continuous_deployment.md`